In [1]:
from huggingface_hub import hf_hub_download
import torch

path = hf_hub_download(repo_id="gpt2", filename="pytorch_model.bin")
sd_hf = torch.load(path, map_location="cpu")

/home/ubuntu/gziz/myllm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/gziz/myllm/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:
sd_hf.keys()

odict_keys(['wte.weight', 'wpe.weight', 'h.0.ln_1.weight', 'h.0.ln_1.bias', 'h.0.attn.bias', 'h.0.attn.c_attn.weight', 'h.0.attn.c_attn.bias', 'h.0.attn.c_proj.weight', 'h.0.attn.c_proj.bias', 'h.0.ln_2.weight', 'h.0.ln_2.bias', 'h.0.mlp.c_fc.weight', 'h.0.mlp.c_fc.bias', 'h.0.mlp.c_proj.weight', 'h.0.mlp.c_proj.bias', 'h.1.ln_1.weight', 'h.1.ln_1.bias', 'h.1.attn.bias', 'h.1.attn.c_attn.weight', 'h.1.attn.c_attn.bias', 'h.1.attn.c_proj.weight', 'h.1.attn.c_proj.bias', 'h.1.ln_2.weight', 'h.1.ln_2.bias', 'h.1.mlp.c_fc.weight', 'h.1.mlp.c_fc.bias', 'h.1.mlp.c_proj.weight', 'h.1.mlp.c_proj.bias', 'h.2.ln_1.weight', 'h.2.ln_1.bias', 'h.2.attn.bias', 'h.2.attn.c_attn.weight', 'h.2.attn.c_attn.bias', 'h.2.attn.c_proj.weight', 'h.2.attn.c_proj.bias', 'h.2.ln_2.weight', 'h.2.ln_2.bias', 'h.2.mlp.c_fc.weight', 'h.2.mlp.c_fc.bias', 'h.2.mlp.c_proj.weight', 'h.2.mlp.c_proj.bias', 'h.3.ln_1.weight', 'h.3.ln_1.bias', 'h.3.attn.bias', 'h.3.attn.c_attn.weight', 'h.3.attn.c_attn.bias', 'h.3.attn.c_pr

In [6]:
sd_hf["h.0.attn.c_attn.weight"].shape

torch.Size([768, 2304])

In [8]:
2304/3

768.0

In [9]:
# Verify every key load_weights_into_gpt() needs is present in sd_hf
n_blocks = 12  # GPT-2 small

expected = {"wte.weight", "wpe.weight", "ln_f.weight", "ln_f.bias"}
for b in range(n_blocks):
    p = f"h.{b}"
    expected.update({
        f"{p}.attn.c_attn.weight", f"{p}.attn.c_attn.bias",
        f"{p}.attn.c_proj.weight", f"{p}.attn.c_proj.bias",
        f"{p}.mlp.c_fc.weight",    f"{p}.mlp.c_fc.bias",
        f"{p}.mlp.c_proj.weight",  f"{p}.mlp.c_proj.bias",
        f"{p}.ln_1.weight",        f"{p}.ln_1.bias",
        f"{p}.ln_2.weight",        f"{p}.ln_2.bias",
    })

actual = set(sd_hf.keys())
missing = expected - actual
extra = actual - expected
print("missing in sd_hf:", sorted(missing))
print("extra in sd_hf  :", sorted(extra))

missing in sd_hf: []
extra in sd_hf  : ['h.0.attn.bias', 'h.1.attn.bias', 'h.10.attn.bias', 'h.11.attn.bias', 'h.2.attn.bias', 'h.3.attn.bias', 'h.4.attn.bias', 'h.5.attn.bias', 'h.6.attn.bias', 'h.7.attn.bias', 'h.8.attn.bias', 'h.9.attn.bias']


In [1]:
from gpt2.train import train_model_simple
import torch
from gpt2.model import GPTModel
from gpt2.configs import model_configs
from gpt2.utils import text_to_token_ids, token_ids_to_text, create_dataloader_v1
from nanochat.dataset import parquets_iter_batched

import tiktoken

# Load data from nanochat parquet shards
MAX_TRAIN_DOCS = 100
MAX_VAL_DOCS = 20

train_texts = []
for batch in parquets_iter_batched(split="train"):
    print(f"Loaded batch of {len(batch)} training documents")
    train_texts.extend(batch)
    if len(train_texts) >= MAX_TRAIN_DOCS:
        train_texts = train_texts[:MAX_TRAIN_DOCS]
        break
train_text = "\n".join(train_texts)

val_texts = []
for batch in parquets_iter_batched(split="val"):
    print(f"Loaded batch of {len(batch)} validation documents")
    val_texts.extend(batch)
    if len(val_texts) >= MAX_VAL_DOCS:
        val_texts = val_texts[:MAX_VAL_DOCS]
        break
val_text = "\n".join(val_texts)


Loaded batch of 1024 training documents
Loaded batch of 1024 validation documents


In [2]:
for batch in parquets_iter_batched(split="train"):
    break

In [3]:
batch

['Protect your business and employee\'s during flu seaon\n\nStarting in 2005, the Center for Disease Control (CDC) established the National Influenza Vaccination week "to highlight the importance of continuing flu vaccination through the holiday season and beyond." In 2017, by the end of November only about 41% of the recommended US population had been vaccinated.\n\nIf you\'ve already gotten the flu, you can still get vaccinated to protect you against other strands of the flu, according to the CDC. People with high risk of complications from the flu include young children, pregnant women, people with certain chronic health conditions and people over the age of 65, and should get vaccinated. Although anyone is at risk of getting the flu, for high-risk people, hospitalization or death is more likely than for others.\n\nAccording to the CDC, "flu vaccination can reduce flu illnesses, doctors\' visits and missed work and school due to flu, as well as prevent flu-related hospitalizations."